In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 25.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.33.1
    Uninstalling huggingface-hub-0.33.1:
      Successfully uninstalled huggingface-hub-0.33.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.5.1 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings("ignore")

2025-08-10 23:41:46.681411: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754869306.938303      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754869307.005016      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train = pd.read_csv('/kaggle/input/twitter-entity-sentiment-analysis/twitter_training.csv')
vaildation = pd.read_csv('/kaggle/input/twitter-entity-sentiment-analysis/twitter_validation.csv')

In [4]:
train.columns = ['ID' , 'ENTITY' , 'label', 'tweet']
vaildation.columns = ['ID' , 'ENTITY' , 'label', 'tweet']

In [5]:
train.drop(['ID' , 'ENTITY'] , axis = 1 , inplace = True)
vaildation.drop(['ID' , 'ENTITY'] , axis = 1 , inplace = True)

In [6]:
train.label.value_counts()

label
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [7]:
vaildation.label.value_counts()

label
Neutral       285
Positive      277
Negative      266
Irrelevant    171
Name: count, dtype: int64

In [8]:
train = train[train.label != 'Irrelevant']

In [9]:
vaildation = vaildation[vaildation.label != 'Irrelevant']

In [10]:
train.label.value_counts()

label
Negative    22542
Positive    20831
Neutral     18318
Name: count, dtype: int64

In [11]:
vaildation.label.value_counts()

label
Neutral     285
Positive    277
Negative    266
Name: count, dtype: int64

In [12]:
train.isnull().sum()

label      0
tweet    571
dtype: int64

In [13]:
train.dropna(inplace=True)

In [14]:
train.isnull().sum()

label    0
tweet    0
dtype: int64

In [15]:
train.duplicated().sum()

3636

In [16]:
train.drop_duplicates(inplace=True)

In [17]:
train.duplicated().sum()

0

In [18]:
vaildation.isnull().sum()

label    0
tweet    0
dtype: int64

In [19]:
vaildation.duplicated().sum()

1

In [20]:
vaildation.drop_duplicates(inplace=True)

In [21]:
vaildation.duplicated().sum()

0

In [22]:
labels = train.label.unique()
label2id = {label: i for i , label in enumerate(labels)}
id2label = {i:label for i , label in enumerate(labels)}

In [23]:
train['sentiment'] = train.label.map(label2id)
vaildation['sentiment'] = vaildation.label.map(label2id)

In [24]:
train_data = Dataset.from_pandas(train[['tweet' , 'sentiment']].rename(columns={'tweet': 'text' , 'sentiment' :'labels'}))
vaildation_data = Dataset.from_pandas(vaildation[['tweet' , 'sentiment']].rename(columns={'tweet': 'text' , 'sentiment' :'labels'}))

In [25]:
from transformers import AutoTokenizer

#model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [26]:
print(train_data.column_names)

['text', 'labels', '__index_level_0__']


In [27]:
def preprocess_function(examples):
    return tokenizer(examples['text'] , padding = 'max_length' , truncation = True , max_length = 128)

tokenized_train_dataset = train_data.map(preprocess_function, batched=True)
tokenized_vaildation_dataset = vaildation_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/57484 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

In [28]:
print(tokenized_train_dataset.column_names)
print(tokenized_vaildation_dataset.column_names)

['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']
['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']


In [29]:
model = AutoModelForSequenceClassification.from_pretrained(model_name ,
                                                           num_labels = len(labels) ,
                                                           id2label = id2label ,
                                                           label2id = label2id ,
                                                           ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    prediction , labels = eval_pred
    prediction = np.argmax(prediction , axis = 1)
    acc = accuracy_score(labels , prediction)
    f1 = f1_score(labels , prediction , average = 'weighted')
    return {'accuracy' : acc , 'f1' : f1}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=500,
    report_to="none",
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train_dataset,
    eval_dataset = tokenized_vaildation_dataset,
    compute_metrics = compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
500,0.763900
1000,0.549000
1500,0.437700


TrainOutput(global_step=1797, training_loss=0.5491002952647859, metrics={'train_runtime': 361.3979, 'train_samples_per_second': 159.06, 'train_steps_per_second': 4.972, 'total_flos': 1903722935675904.0, 'train_loss': 0.5491002952647859, 'epoch': 1.0})

In [30]:
from transformers import pipeline
import os
os.environ["WANDB_DISABLED"] = "true"

classifier = pipeline("sentiment-analysis" , model = trainer.model , tokenizer = tokenizer)

new_texts = [
    "This is a fantastic movie; the acting and plot were amazing.",
    "The food at this restaurant is absolutely delicious.",
    "The service was incredibly slow and the food was cold.",
    "I was very disappointed with the outcome of the project.",
    "I finished reading the report this morning.",
    "She walked down the street towards the bus stop."
]

results = classifier(new_texts)

for text , result in zip(new_texts , results):
    print(f'{text}')
    print(f"Sentiment: {result['label']} , (Score: {result['score']})")
    print()

Device set to use cuda:0


This is a fantastic movie; the acting and plot were amazing.
Sentiment: Positive , (Score: 0.9765229821205139)

The food at this restaurant is absolutely delicious.
Sentiment: Positive , (Score: 0.9723697900772095)

The service was incredibly slow and the food was cold.
Sentiment: Negative , (Score: 0.4941919445991516)

I was very disappointed with the outcome of the project.
Sentiment: Negative , (Score: 0.9718616008758545)

I finished reading the report this morning.
Sentiment: Positive , (Score: 0.48909103870391846)

She walked down the street towards the bus stop.
Sentiment: Neutral , (Score: 0.6968275308609009)



In [31]:
output_save_dir = "twitter_sentiment_model"
trainer.save_model(output_save_dir)
tokenizer.save_pretrained(output_save_dir)

('twitter_sentiment_model/tokenizer_config.json',
 'twitter_sentiment_model/special_tokens_map.json',
 'twitter_sentiment_model/vocab.txt',
 'twitter_sentiment_model/added_tokens.json',
 'twitter_sentiment_model/tokenizer.json')

In [32]:
!zip -r twitter_sentiment_model.zip twitter_sentiment_model

  adding: twitter_sentiment_model/ (stored 0%)
  adding: twitter_sentiment_model/training_args.bin (deflated 52%)
  adding: twitter_sentiment_model/config.json (deflated 48%)
  adding: twitter_sentiment_model/tokenizer.json (deflated 71%)
  adding: twitter_sentiment_model/special_tokens_map.json (deflated 42%)
  adding: twitter_sentiment_model/tokenizer_config.json (deflated 75%)
  adding: twitter_sentiment_model/vocab.txt (deflated 53%)
  adding: twitter_sentiment_model/model.safetensors (deflated 8%)
